In [13]:
from web3 import Web3
import json

In [14]:
# Load config
with open("config.json") as f:
    config = json.load(f)

w3 = Web3(Web3.HTTPProvider(config["blockchain"]["rpc_url"]))
print("Connected:", w3.is_connected())

stablecoin_address = config["blockchain"]["stablecoin_address"]
p2p_market_address = config["blockchain"]["p2p_market_address"]

# Minimal ERC-20 ABI — only what we need for checking allowances
erc20_abi = [
    {
        "constant": True,
        "inputs": [
            {"name": "owner", "type": "address"},
            {"name": "spender", "type": "address"}
        ],
        "name": "allowance",
        "outputs": [{"name": "", "type": "uint256"}],
        "type": "function"
    },
    {
        "constant": True,
        "inputs": [{"name": "account", "type": "address"}],
        "name": "balanceOf",
        "outputs": [{"name": "", "type": "uint256"}],
        "type": "function"
    }
]

stablecoin = w3.eth.contract(address=Web3.to_checksum_address(stablecoin_address), abi=erc20_abi)

Connected: True


In [15]:
household_1 = "0x4022d2250AB1E76d3fcbCcf18d39656f4559b83c"
household_2 = "0xF49153d700AD86CA224f7B2064F541278FE1c320"
household_3 = "0xC416DDdD40c28eAfE48275f74B4E4Bdee4E3e75b"


allowance_1 = stablecoin.functions.allowance(household_1, p2p_market_address).call()
print(allowance_1)  # should be a huge number (2**256 - 1) if approved, 0 if not

allowance_2 = stablecoin.functions.allowance(household_2, p2p_market_address).call()
print(allowance_2)  # should be a huge number (2**256 - 1) if approved, 0 if not

allowance_3 = stablecoin.functions.allowance(household_3, p2p_market_address).call()
print(allowance_3)  # should be a huge number (2**256 - 1) if approved, 0 if not

50
50
0


In [16]:
for household in config["households"]:
    addr = Web3.to_checksum_address(household["address"])
    allowance = stablecoin.functions.allowance(addr, p2p_market_address).call()
    balance = stablecoin.functions.balanceOf(addr).call()
    print(f"{household.get('name', addr)}: allowance={allowance}, balance={balance}")

0x4022d2250AB1E76d3fcbCcf18d39656f4559b83c: allowance=50, balance=100000000
0xF49153d700AD86CA224f7B2064F541278FE1c320: allowance=50, balance=5000000


In [19]:
from web3 import Web3
from pathlib import Path
import json
import os
from dotenv import load_dotenv

load_dotenv()

CONFIG_PATH = Path("config.json")   # adjust path if your notebook lives elsewhere
ABI_DIR = Path("abi")

with open(CONFIG_PATH) as f:
    config = json.load(f)

bc = config["blockchain"]
w3 = Web3(Web3.HTTPProvider(bc["rpc_url"]))
print("Connected:", w3.is_connected())

# Load P2PEnergyMarket ABI
with open(ABI_DIR / "OracleStorage.json") as f:
    oracle_abi = json.load(f)["abi"]


oracleStorage = w3.eth.contract(
    address=Web3.to_checksum_address(bc["oracle_storage_address"]),
    abi=oracle_abi
)

oracleStorage.authorizeOracle(os.getenv("ORACLE_WALLET_ADDR"))

Connected: True


AttributeError: 'Contract' object has no attribute 'authorizeOracle'

In [20]:
from web3 import Web3
from pathlib import Path
import json
import os
from dotenv import load_dotenv

load_dotenv()

CONFIG_PATH = Path("config.json")   # adjust path if your notebook lives elsewhere
ABI_DIR = Path("abi")

with open(CONFIG_PATH) as f:
    config = json.load(f)

bc = config["blockchain"]
w3 = Web3(Web3.HTTPProvider(bc["rpc_url"]))
print("Connected:", w3.is_connected())

# Load P2PEnergyMarket ABI
with open(ABI_DIR / "P2PEnergyMarket.json") as f:
    p2p_abi = json.load(f)["abi"]


p2pMarket = w3.eth.contract(
    address=Web3.to_checksum_address(bc["p2p_market_address"]),
    abi=p2p_abi
)

for household in config["households"]:
    p2pMarket.registerHousehold(household["address"])
    


Connected: True


AttributeError: 'Contract' object has no attribute 'registerHousehold'